In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
basedf = pd.read_csv('/data/zsl23/preparation/basedf.csv', sep=",", header = 0, index_col=None)

In [15]:
basedf.head()

,#id,redshift,distance,galex.FUV,galex.FUV_err,galex.NUV,galex.NUV_err,sloan.sdss.u,sloan.sdss.u_err,sloan.sdss.g,...,herschel.pacs.green,herschel.pacs.green_err,herschel.pacs.red,herschel.pacs.red_err,herschel.spire.PSW,herschel.spire.PSW_err,herschel.spire.PMW,herschel.spire.PMW_err,herschel.spire.PLW,herschel.spire.PLW_err
0,ESO149-013,0,20.248498,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,49.629813,82.990714,7.899533,58.638459,7.301990,34.582998
1,NGC0007,0,20.606305,2.792007,0.130765,3.131622,0.109649,NaN,NaN,NaN,...,-585.993205,550.268549,479.281338,561.583315,419.146475,108.798696,343.955863,66.855994,168.743455,48.714171
2,ESO410-005,0,1.940886,0.169843,0.015266,0.505197,0.029494,NaN,NaN,NaN,...,606.719823,298.355089,575.689527,351.670292,49.891475,87.069055,15.661851,65.672827,9.789736,33.119280
3,IC0010,0,0.794328,NaN,NaN,82636.651201,332637.806656,NaN,NaN,NaN,...,216045.743398,37151.678713,214645.823529,118767.051682,110582.864955,59084.049307,52926.952540,40419.362730,21082.391740,14078.461687
4,NGC0115,0,28.707818,3.101272,0.142196,3.916178,0.110150,NaN,NaN,NaN,...,896.019859,600.566138,2157.135283,488.568231,912.792408,129.122311,495.308969,97.804529,259.551036,60.057107


In [16]:
# assigning columns/filters to wavelength ranges
galex_cols   = ['galex.FUV', 'galex.NUV'] # UV
sdss_cols    = ['sloan.sdss.u', 'sloan.sdss.g', 'sloan.sdss.r', 'sloan.sdss.i', 'sloan.sdss.z'] # VIS
nir_cols_tmass = ['2mass.J', '2mass.H', '2mass.Ks'] # NIR
mir_cols = ['wise.W1', 'wise.W2', 'wise.W3',
            'spitzer.irac.I1', 'spitzer.irac.I2', 'spitzer.irac.I3', 'spitzer.irac.I4'] # MIR
fir_cols_belowspire = ['wise.W4', 'spitzer.mips.24mu', 'spitzer.mips.70mu', 'spitzer.mips.160mu',
            'herschel.pacs.blue', 'herschel.pacs.green', 'herschel.pacs.red'] # near end of the FIR
spire_cols   = ['herschel.spire.PSW', 'herschel.spire.PMW', 'herschel.spire.PLW'] # far end of the FIR

In [17]:
# some functions to create criteria
def anypresent(cols):
    """Boolean mask: True where at least one of the given columns is not NaN."""
    return basedf[cols].notna().any(axis=1) # axis=1 looks at the columns. so for each row, any of the columns is not NaN, 
    # then the entire row is true.

def allpresent(cols):
    """Boolean mask: True only where all of the given columns is not NaN."""
    return basedf[cols].notna().all(axis=1) 
    
# pandas.notna() identifies non-missing values in a Series/DataFrame. It returns a Boolean mask (True for present values, False for NaNs).

def count_present(cols):
    """Row count of non NaN columns among the given list."""
    return basedf[cols].notna().sum(axis=1) # for each row, sum up the number of columns that are true.

In [18]:
# criteria
crit_galex = anypresent(galex_cols)
crit_sdss  = allpresent(sdss_cols)
crit_nir = allpresent(nir_cols_tmass)

crit_mir = count_present(mir_cols) >= 3
crit_fir = count_present(fir_cols_belowspire) >= 3
crit_spire = allpresent(spire_cols)

# combine the criteriaglobalgaldf[value_cols] = globalgaldf[value_cols] * 1000  # convert Jy -> mJy
final_mask = (crit_galex & crit_sdss & crit_nir & crit_mir & crit_fir & crit_spire)

In [19]:
print(f"  GALEX (FUV or NUV):              {crit_galex.sum()}")
print(f"  SDSS (all 5 bands):              {crit_sdss.sum()}")
print(f"  NIR (all 3 bands):               {crit_nir.sum()}")
print(f"  MIR >=3 bands:                   {crit_mir.sum()}")
print(f"  FIR >=3 bands:                   {crit_fir.sum()}")
print(f"  SPIRE (all 3 bands):             {crit_spire.sum()}")
print()
print(f"FINAL count (all criteria met): {final_mask.sum()}")

  GALEX (FUV or NUV):              748
  SDSS (all 5 bands):              578
  NIR (all 3 bands):               768
  MIR >=3 bands:                   777
  FIR >=3 bands:                   623
  SPIRE (all 3 bands):             756

FINAL count (all criteria met): 417


In [20]:
basegal = basedf[final_mask]

In [21]:
mask1 = (basegal['spitzer.mips.160mu_err'] < 0) & (basegal['herschel.pacs.red'].notna())
basegal.loc[mask1, ['spitzer.mips.160mu', 'spitzer.mips.160mu_err']] = 'NaN'

mask2 = (basegal['spitzer.mips.70mu_err'] < 0) & (basegal['herschel.pacs.blue'].notna())
basegal.loc[mask2, ['spitzer.mips.70mu', 'spitzer.mips.70mu_err']] = 'NaN'

/tmp/ipykernel_315778/3096066530.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NaN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  basegal.loc[mask1, ['spitzer.mips.160mu', 'spitzer.mips.160mu_err']] = 'NaN'
/tmp/ipykernel_315778/3096066530.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NaN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  basegal.loc[mask1, ['spitzer.mips.160mu', 'spitzer.mips.160mu_err']] = 'NaN'
/tmp/ipykernel_315778/3096066530.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'NaN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  basegal.loc[mask2, ['spitzer.mips.70mu', 'spitzer.mips

In [22]:
ngc4450 = basegal[basegal['#id']=='NGC4550']
ngc4450['spitzer.mips.160mu']

550    NaN
Name: spitzer.mips.160mu, dtype: object

In [23]:
basegal.to_csv('basemJynan_new.txt' ,sep=' ', index=False, na_rep='NaN') # to be used in CIGALE

In [24]:
# there is one problematic galaxy PGC028759, need to remove it from analysis
# mask = basegalnan['#id'].str.contains(r"PGC028759", na=False)
# basegalnan = basegalnan[~mask]